# Google Drive Video Transcriber
Transcribe public Google Drive videos using **faster-whisper large-v3** on a T4 GPU.

### Usage
1. **Run Cell 1 once** — installs dependencies and downloads the model (~3GB). Takes a few minutes.
2. **Run Cell 2 as many times as you want** — paste a new link each time. The model stays loaded in memory.

---

In [ ]:
#@title **Cell 1 — Run Once: Install & Load Model**

# Install dependencies
!pip install -q faster-whisper gdown

import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

# Load model into GPU memory (this is what we want to do only once)
from faster_whisper import WhisperModel

print("Downloading and loading large-v3 model (first run will take a few minutes)...")
model = WhisperModel("large-v3", device="cuda", compute_type="float16")
print("Model loaded and ready.")

In [ ]:
#@title **Cell 2 — Transcribe a Video (run repeatedly with different links)**

#@markdown Paste the public Google Drive share link below:
video_url = "https://drive.google.com/file/d/YOUR_FILE_ID_HERE/view?usp=sharing" #@param {type:"string"}

#@markdown Output filename (without extension):
output_name = "transcript" #@param {type:"string"}

import re, os, time, gdown, subprocess

# Extract file ID from various Drive URL formats
def extract_file_id(url):
    patterns = [
        r'/file/d/([a-zA-Z0-9_-]+)',
        r'id=([a-zA-Z0-9_-]+)',
        r'^([a-zA-Z0-9_-]{20,})$',
    ]
    for p in patterns:
        m = re.search(p, url.strip())
        if m:
            return m.group(1)
    raise ValueError(f"Could not extract a file ID from: {url}")

file_id = extract_file_id(video_url)
download_url = f"https://drive.google.com/uc?id={file_id}"
video_path = "/content/video_input"
audio_path = "/content/audio_temp.wav"
txt_path = f"/content/{output_name}.txt"

# Clean up previous run
for f in [video_path, audio_path]:
    if os.path.exists(f):
        os.remove(f)

# Step 1: Download
print("[1/3] Downloading video from Google Drive...")
t0 = time.time()
gdown.download(download_url, video_path, quiet=False, fuzzy=True)
if not os.path.exists(video_path):
    raise RuntimeError("Download failed. Is the link public and valid?")
size_mb = os.path.getsize(video_path) / (1024 * 1024)
print(f"    Downloaded {size_mb:.1f} MB in {time.time() - t0:.1f}s")

# Step 2: Extract audio
print("[2/3] Extracting audio (mono 16kHz WAV)...")
t0 = time.time()
result = subprocess.run(
    ["ffmpeg", "-y", "-i", video_path, "-vn",
     "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", audio_path],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("ffmpeg failed — is this a valid video file?")
print(f"    Audio extracted in {time.time() - t0:.1f}s")

os.remove(video_path)

# Step 3: Transcribe
print("[3/3] Transcribing with faster-whisper large-v3 (GPU)...")
t0 = time.time()
segments, info = model.transcribe(audio_path, language="en", beam_size=5)

lines = []
for seg in segments:
    lines.append(seg.text.strip())

transcript = "\n".join(lines)
elapsed = time.time() - t0
print(f"    Transcription done in {elapsed:.1f}s (audio duration: {info.duration:.1f}s)")

# Clean up audio
os.remove(audio_path)

# Save
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(transcript)
print(f"\nSaved to: {txt_path}")
print("="*60)
print(transcript)